# Gold dimension: `dim_project`

## Purpose

One descriptive row per DPWH contract for `fact_project_snapshot`, the dashboard, and Genie.

## Grain

One contract per source system. Uniqueness: `source_system`, `contract_id`.

## Keys

Primary key `project_key`. Recipe: `XXHASH64` of `PROJECT`, the source system, and the Contract ID (D-33).

## Key 0 member

None. Every fact row comes from a real `silver_project` row (D-35).

## Inputs

- `02-silver.silver_project`
- `02-silver.silver_project_source_match` (exact Contract ID matches only)
- `04-validation.silver_dq_results`
- `01-bronze.load_log` (load time only)

## Outputs

`03-gold.dim_project`, loaded with `MERGE`. Rows are never deleted.

## Not done here

No category mapping, because the taxonomy comes from Silver config. Flood-list cost is never added to the DPWH budget (D-04).

## 1. Confirm upstream safety and select versions

**What:** check that `silver_project` and `silver_project_source_match` are each one validated run. Check that the match was built from the current project run.

**Why:** a match from an older project run could mark the wrong projects as flood-list members.

**Protects:** every check stops the notebook before any write.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

DECLARE OR REPLACE VARIABLE project_source_system STRING DEFAULT 'DPWH';
DECLARE OR REPLACE VARIABLE project_rule_version STRING DEFAULT 'gold-dim-project-v1';

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT source_ingest_run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT project_key),
    'STOP: silver_project must be nonempty and unique in one selected run'
)
FROM `02-silver`.silver_project;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT dq.table_name) = 2
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver project validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name IN ('silver_project', 'silver_dpwh_project_component')
  AND dq.pipeline_run_id = (SELECT MAX(project.run_id) FROM `02-silver`.silver_project AS project);

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT target_project_run_id) = 1
    AND MAX(target_project_run_id) = (SELECT MAX(project.run_id) FROM `02-silver`.silver_project AS project),
    'STOP: silver_project_source_match must be one run built from the current silver_project run'
)
FROM `02-silver`.silver_project_source_match;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT dq.table_name) = 2
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver source-reconciliation validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name IN ('silver_project_source_match', 'silver_flood_control_component')
  AND dq.pipeline_run_id = (
      SELECT MAX(source_match.run_id) FROM `02-silver`.silver_project_source_match AS source_match
  );

CREATE OR REPLACE TEMPORARY VIEW project_load_lineage AS
SELECT run_id AS source_ingest_run_id, source_version,
       COALESCE(completed_at, started_at) AS source_load_ts
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY run_id
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS audit_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'dpwh_projects'
      AND status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
)
WHERE audit_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW selected_project_versions AS
WITH project_context AS (
    SELECT
        MAX(run_id) AS project_run_id,
        MAX(source_ingest_run_id) AS project_source_ingest_run_id
    FROM `02-silver`.silver_project
),
flood_list_context AS (
    SELECT
        MAX(run_id) AS flood_list_run_id,
        MAX(COALESCE(source_version, source_snapshot_id)) AS flood_list_version,
        MAX(source_load_ts) AS flood_list_load_ts
    FROM `02-silver`.silver_project_source_match
)
SELECT
    project_context.project_run_id,
    flood_list_context.flood_list_run_id,
    flood_list_context.flood_list_version,
    SHA2(CONCAT_WS(
        '|', project_context.project_run_id, flood_list_context.flood_list_run_id, project_rule_version
    ), 256) AS project_dim_run_id,
    GREATEST(load.source_load_ts, flood_list_context.flood_list_load_ts) AS source_load_ts
FROM project_context
CROSS JOIN flood_list_context
LEFT JOIN project_load_lineage AS load
    ON project_context.project_source_ingest_run_id = load.source_ingest_run_id;

## 2. Collect flood-list members

**What:** list each canonical project that has an exact Contract ID match in the official flood-control list.

**Why:** only `MATCHED_EXACT_CONTRACT_ID` rows count. Unmatched flood-list IDs never create a project here.

**Protects:** one project must appear at most once, so flood-list membership can never duplicate a project.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW flood_list_members AS
SELECT
    source_match.matched_project_key,
    COUNT(*) AS flood_list_match_rows
FROM `02-silver`.silver_project_source_match AS source_match
WHERE source_match.match_status = 'MATCHED_EXACT_CONTRACT_ID'
GROUP BY source_match.matched_project_key;

SELECT ASSERT_TRUE(
    COUNT_IF(flood_list_match_rows <> 1) = 0,
    'STOP: a canonical project matches more than one flood-list Contract ID group'
)
FROM flood_list_members;

## 3. Create the table

**What:** create `03-gold.dim_project` with the column names, types, and order in `docs/gold_model.md`.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.dim_project (
    project_key BIGINT NOT NULL COMMENT 'Surrogate key. XXHASH64 of the label PROJECT, the source system, and the Contract ID.',
    contract_id STRING COMMENT 'DPWH Contract ID exactly as published.',
    source_system STRING COMMENT 'Source system of the project. Always DPWH.',
    project_description STRING COMMENT 'Project description. NULL when the source rows disagree.',
    raw_category STRING COMMENT 'DPWH category exactly as published. It mixes work categories and funding codes, such as GAA 2025 SSP.',
    source_infra_type STRING COMMENT 'Source infrastructure type. NULL because DPWH has no infrastructure-type column. The team has not chosen a source field.',
    standardized_sector STRING COMMENT 'Team-approved sector from the category taxonomy. NULL until a taxonomy is approved.',
    infra_type_mapping_status STRING COMMENT 'Category mapping result: MAPPED, UNMAPPED, MISSING_SOURCE_CATEGORY, or CONFLICTING_SOURCE_CATEGORY.',
    taxonomy_version STRING COMMENT 'Approved category taxonomy version. NO_APPROVED_TAXONOMY when none applies.',
    is_dpwh_flood_related BOOLEAN COMMENT 'True when the approved taxonomy marks the category as flood related. NULL until a taxonomy is approved.',
    is_in_official_flood_list BOOLEAN COMMENT 'True when the Contract ID exactly matches the official flood-control list. Membership only. Flood-list cost is never added to the budget.',
    flood_list_match_status STRING COMMENT 'MATCHED_EXACT_CONTRACT_ID or NOT_IN_FLOOD_LIST.',
    flood_list_version STRING COMMENT 'Version of the flood-control list that was searched. The snapshot ID when no publisher version exists.',
    funding_source STRING COMMENT 'Source of funds as published. NULL when the source rows disagree.',
    implementing_office STRING COMMENT 'Implementing District Engineering Office. NULL until Silver carries the DPWH deo field.',
    contractor_name STRING COMMENT 'Contractor as published. NULL when the source rows disagree.',
    infra_year INT COMMENT 'Infrastructure program year. NULL when the source value is not a whole year.',
    run_id STRING COMMENT 'Deterministic Gold run ID. Hash of the Silver project run, the source-match run, and the Gold rule version.',
    source_load_ts TIMESTAMP COMMENT 'Latest Bronze load time of the DPWH snapshot and the flood-control list behind this row.'
)
USING DELTA
COMMENT 'Gold project dimension. One row per DPWH contract with descriptive attributes, category mapping status, and official flood-list membership.';

## 4. Build and merge the projects

**What:** build one row per canonical project and merge by `project_key`.

**Why:**

- `infra_year` uses `TRY_CAST`, so a shifted source value becomes `NULL` instead of stopping the run (D-37).
- `infra_type_mapping_status` explains a `NULL` sector: no approved mapping, a missing source category, or conflicting source categories (D-37).

**Expected validation:** one row per canonical project, and the key is unique.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dim_project_source AS
SELECT
    XXHASH64(CONCAT_WS('|', 'PROJECT', project_source_system, project.contract_id)) AS project_key,
    project.contract_id,
    project_source_system AS source_system,
    project.project_description_raw AS project_description,
    project.category_raw AS raw_category,
    CAST(NULL AS STRING) AS source_infra_type,
    project.standardized_sector,
    CASE
        WHEN project.standardized_sector IS NOT NULL THEN 'MAPPED'
        WHEN project.category_resolution_status = 'CONFLICT' THEN 'CONFLICTING_SOURCE_CATEGORY'
        WHEN project.category_resolution_status = 'MISSING' THEN 'MISSING_SOURCE_CATEGORY'
        ELSE 'UNMAPPED'
    END AS infra_type_mapping_status,
    COALESCE(project.taxonomy_version, 'NO_APPROVED_TAXONOMY') AS taxonomy_version,
    CAST(NULL AS BOOLEAN) AS is_dpwh_flood_related,
    flood.matched_project_key IS NOT NULL AS is_in_official_flood_list,
    CASE
        WHEN flood.matched_project_key IS NOT NULL THEN 'MATCHED_EXACT_CONTRACT_ID'
        ELSE 'NOT_IN_FLOOD_LIST'
    END AS flood_list_match_status,
    selected.flood_list_version,
    project.source_of_funds AS funding_source,
    CAST(NULL AS STRING) AS implementing_office,
    project.contractor AS contractor_name,
    TRY_CAST(project.infra_year AS INT) AS infra_year,
    selected.project_dim_run_id AS run_id,
    selected.source_load_ts
FROM `02-silver`.silver_project AS project
CROSS JOIN selected_project_versions AS selected
LEFT JOIN flood_list_members AS flood
    ON project.project_key = flood.matched_project_key;

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_project)
    AND COUNT(*) = COUNT(DISTINCT project_key)
    AND COUNT(*) = COUNT(DISTINCT NAMED_STRUCT('source_system', source_system, 'contract_id', contract_id)),
    'STOP: dim_project must have exactly one unique row per canonical project'
)
FROM dim_project_source;

MERGE INTO `03-gold`.dim_project AS target
USING dim_project_source AS source
ON target.project_key = source.project_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

# Summary

Created or refreshed `03-gold.dim_project` with one row per DPWH contract. Attributes come from `silver_project`, and flood-list membership comes from exact Contract ID matches. Unmapped categories and missing source fields stay visible. The query below shows mapping and flood-list coverage.

Coverage: the query below shows projects by mapping status and flood-list status, against the total. Known limits: no taxonomy is approved, so `standardized_sector` and `is_dpwh_flood_related` are `NULL` and portfolio measures stay blocked. `source_infra_type` and `implementing_office` are `NULL` until Silver publishes them.

Next: the Gold facts.

In [ ]:
%sql
SELECT
    infra_type_mapping_status,
    flood_list_match_status,
    COUNT(*) AS projects,
    SUM(COUNT(*)) OVER () AS total_projects,
    COUNT_IF(infra_year IS NULL) AS projects_without_infra_year
FROM `03-gold`.dim_project
GROUP BY ALL
ORDER BY projects DESC;